# Introduction to Molecular Mechanics 

---

### High-Level Questions
- What drives the binding of ligands to proteins?
- How does a protein fold into it's native state?
- How does molecular behavior at the atomic scale translate to bulk thermodynamic properties? <br>

![Lignad Binding](../assets/images/LigandBinding.png) 
**citation**<br>
Sohraby F, Nunes-Alves A
Advances in computational methods for ligand binding kinetics
Trends in Biochemical Sciences, 2022; 48, 437-449 

[Ligand Binding Molecular Dynamics](https://www.youtube.com/shorts/G48pFBgbqPM)


----

### Quantum Mechanics 
Quantum mechanics governs how particles evolve over time at the molecular and subatomic levels. In other words, classical physics breaks down at these lengths scales.

$i\hbar\frac{\partial \Psi}{\partial t} = \hat{H}\Psi$ : Time dependent Schrodinger equation<br>
$\hat{H}\Psi = E\Psi$ : Time independent Schrodinger equation <br>

$\hat{H} = \hat{K} + \hat{V}$ <br>
The Hamiltonian operator represents the total energy of a system, which is the sum of the kinetic energy operator plus the potential energy operator<br>

Even with our best and most accurate quantum chemical approximations, simulating 10s-100s atoms is impractical for most research labs. <br>


In [82]:
import numpy as np
import plotly.graph_objects as go
from scipy.constants import physical_constants
from scipy.ndimage import map_coordinates
from skimage import measure

BOHR = physical_constants["Bohr radius"][0] * 1e10


def read_cube(path):
    """-> (data, origin, voxel_vectors, atom_xyz), atomic units."""
    with open(path) as f:
        head = [f.readline() for _ in range(6)]
        nat = abs(int(head[2].split()[0]))
        grid = np.array([ln.split() for ln in head[3:6]], float)
        atoms = np.array([f.readline().split() for _ in range(nat)], float)
        data = np.array(f.read().split(), float).reshape(grid[:, 0].astype(int))
    return data, np.array(head[2].split()[1:4], float), grid[:, 1:], atoms[:, 2:5]


def esp_surface(density_cube, esp_cube, isoval=0.002, opacity=1.0):
    dens, origin, vecs, atoms = read_cube(density_cube)
    verts, tris, *_ = measure.marching_cubes(dens, level=isoval)     # shape from density
    esp = map_coordinates(read_cube(esp_cube)[0], verts.T, order=1)  # colour from ESP
    xyz = (origin + verts @ vecs) * BOHR
    lim = np.percentile(np.abs(esp), 98)

    fig = go.Figure(go.Mesh3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        i=tris[:, 0], j=tris[:, 1], k=tris[:, 2],
        intensity=esp, colorscale='Turbo_r', cmin=-lim, cmax=lim, opacity=opacity,
        colorbar_title="ESP<br>(a.u.)"))
    fig.update_layout(width=700, height=600, scene=dict(
        aspectmode="data", xaxis_visible=False,
        yaxis_visible=False, zaxis_visible=False))
    return fig


esp_surface("benzene_density.cube", "benzene_esp.cube").show()

---

# Molecular Mechanics
<span style="color: red;">**Can we come up with simple classical approximations to model the behavior of molecules?**.</span><br>

**Classical Force Fields**:
- Various "flavors" that express $V(r)$ differently
  - We will go over the most common functional forms for biological systems 
- Define what common chemical environments may behave similarly.
  - Does the oxygen atom in a ketone behave the same as in an amide? <br>
  ![Lignad Binding](../assets/images/ketone_vs_amide.png)


- Fit simple functional forms to quantum mechanical and experimental data
  - Each terms has a set of parameters that changes the "shape" of the function to an extent

**Modern approaches**: 
- Machine learning force fields 
- Using machine learning algorithms to define the chemical environment
- Using machine learning algorithms for training classical force fields 

**First: Define the total potential energy of the system**
- Sum valence (strong/short range) terms and nonbonded (weak/long range) terms <br>
    $U(r_{1}, r_{2}, ..., r_{N}) = \sum valence(b_1, b_2, ..., b_N) + \sum nonbonded(nb_1, nb_2, ..., nb_N)$



In [83]:
import numpy as np
import plotly.graph_objects as go

# 1. Define your specific function
def f(x, y):
    return (np.sin(1.2 * x) * np.cos(1.1* y))

# 2. Build the coordinate matrix grid
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)
    
# 3. Generate the 3D plot
fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale='Viridis', showscale=False)])
fig.update_layout(
    title='Potential Energy Surface',
    autosize=False,
    width=800,
    height=700,
    margin=dict(l=0, r=0, t=50, b=0),
    scene=dict(
        xaxis=dict(title='Coordinate 1', showticklabels=False),
        yaxis=dict(title='Coordinate 2', showticklabels=False),
        zaxis=dict(title='Potential Energy', showticklabels=False),
        aspectratio=dict(x=1, y=1, z=0.7),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
    ),
)
fig.show()

---

## Valence Terms / Intramolecular Terms (Strong/Short Range)

<span style="color: red;">**Two Body Term:**</span><br>
How do we describe a chemical bond? 

In [85]:
import numpy as np
import nglview as nv
from ase import Atoms
from rdkit import Chem
from rdkit.Chem import AllChem 
from rdkit.Chem import rdMolTransforms


mol = Chem.MolFromSmiles('O')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol)
base_conf = mol.GetConformer(0)

# Linear scan distances (Forward scan)
forward_scan = np.linspace(0.80, 1.20, 15)  # Stretch from 0.8Å to 1.2Å
single_sweep = np.concatenate([forward_scan, forward_scan[::-1]]) 
full_scan = np.tile(single_sweep, 10)

ase_frames = []
for i, length in enumerate(full_scan):
    new_conf = Chem.Conformer(base_conf)
    rdMolTransforms.SetBondLength(new_conf, 0, 1, length)
    
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    positions = new_conf.GetPositions()
    ase_frames.append(Atoms(symbols=symbols, positions=positions))

trajectory = nv.ASETrajectory(ase_frames)
view = nv.NGLWidget(trajectory)
view.layout.height = '200px'   

view.center()
view

NGLWidget(layout=Layout(height='200px'), max_frame=299)

**Here we will use a Morse potential as our reference bond potential.**<br>
**The Morse potential captures the essential physics to describe bond breaking.** 

$U(r) = D_{e}(1 - e^{-\alpha(r - r_{eq})})^2$ <br>

$D_{e}$ = well-depth<br>
$\alpha$ = width of the potential <br>
$r_{eq}$ = equilibrium bond distance



In [86]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import plotly.graph_objects as go

def compute_morse_potential(distances, alpha=1.8, r_eq= 2.0, well_depth=20.0):
    """
    computer a morse potential given an array of distances

    :param distances: bond distances
    :param alpha: 
    :param r_eq: equilibrium distance in Angstroms
    :param well_depth: dissociation energy in kcal/mol

    :return energies: kcal/mol
    """
    return well_depth*(1 - np.exp(-alpha*(distances - r_eq)))**2

distances = np.linspace(1.6, 5.0, 100)
v_curve = compute_morse_potential(distances)

req = 2.0
delta = 0.2
n_points = 40 
min_r = req - delta
max_r = req + delta
displacement = req / 2.0
forward_r = np.linspace(min_r, max_r, n_points)
single_oscillation_r = np.concatenate([forward_r, forward_r[::-1]])
samples_distances = np.tile(single_oscillation_r, 4)

forward_a = np.linspace(displacement-(delta/2), displacement+(delta/2), n_points)[::-1] # Starts flipped
single_oscillation_a = np.concatenate([forward_a, forward_a[::-1]])
atom_a_x = np.tile(single_oscillation_a, 4)

forward_b = np.linspace((displacement+req)-(delta/2), (displacement+req)+(delta/2), n_points)
single_oscillation_b = np.concatenate([forward_b, forward_b[::-1]])
atom_b_x = np.tile(single_oscillation_b, 4)

# Compute energy based on the updated structural distances loop
sampled_energies = compute_morse_potential(samples_distances)


fig = go.Figure(
    data=[
        go.Scatter(x=distances, y=v_curve, mode='lines', line=dict(color='gray', dash='dash'), showlegend=False),
        go.Scatter(x=samples_distances, y=sampled_energies, mode='markers', name='Energy State'),
        go.Scatter(x=atom_a_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 1'),
        go.Scatter(x=atom_b_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 2')
    ],
    frames=[go.Frame(data=[
        go.Scatter(),
        go.Scatter(x=[samples_distances[i]], y=[sampled_energies[i]]),
        go.Scatter(x=[atom_a_x[i]]),
        go.Scatter(x=[atom_b_x[i]])
    ]) for i, _ in enumerate(samples_distances)]
)

fig.update_layout(
    xaxis=dict(title="Bond Length (Angstroms)", range=[1.6, 5.0]),
    xaxis2=dict(title="Position: x", range=[0, 17.0], side='top', overlaying='x', showgrid=False),
    yaxis=dict(title="Energy (kcal/mol)", range=[-0.8, 20.0]),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=25, redraw=False))])])]
)

fig.show()


<span style="color: red;">**What if we focus on the minium?**</span><br>

In [87]:


fig = go.Figure(
    data=[
        go.Scatter(x=distances, y=v_curve, mode='lines', line=dict(color='gray', dash='dash'), showlegend=False),
        go.Scatter(x=samples_distances, y=sampled_energies, mode='markers', name='Energy State'),
        go.Scatter(x=atom_a_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 1'),
        go.Scatter(x=atom_b_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 2')
    ],
    frames=[go.Frame(data=[
        go.Scatter(),
        go.Scatter(x=[samples_distances[i]], y=[sampled_energies[i]]),
        go.Scatter(x=[atom_a_x[i]]),
        go.Scatter(x=[atom_b_x[i]])
    ]) for i, _ in enumerate(samples_distances)]
)

fig.update_layout(
    xaxis=dict(title="Bond Length (Angstroms)", range=[1.8, 2.2]),
    xaxis2=dict(title="Position: x", range=[0.9, 3.1], side='top', overlaying='x', showgrid=False),
    yaxis=dict(title="Energy (kcal/mol)", range=[0.0, 4.0]),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=25, redraw=False))])])]
)

fig.show()


**Typically  the bond stretch term is expressed as a harmonic potential**<br>

$U_{bond} = \frac{1}{2}k(r - r_{eq})^2$





In [88]:
def compute_bond_stretch(distances, k=80.0, r_eq=2.0):
    return k * (distances - r_eq)**2

bond_stretch_ene_20 = compute_bond_stretch(distances, k=20.0)
bond_stretch_ene_80 = compute_bond_stretch(distances, k=80.0)

fig = go.Figure(
    data=[
        go.Scatter(x=distances, y=v_curve, mode='lines', line=dict(color='gray', dash='dash'), name='Reference'),
        go.Scatter(x=distances, y=bond_stretch_ene_20, mode='lines', line=dict(color='red', dash='dash'), name='Harmonic Potential k=20'),
        go.Scatter(x=distances, y=bond_stretch_ene_80, mode='lines', line=dict(color='black', dash='dash'), name='Harmonic Potential k=80'),
        go.Scatter(x=samples_distances, y=sampled_energies, mode='markers', name='Energy State'),
        go.Scatter(x=atom_a_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 1'),
        go.Scatter(x=atom_b_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 2')
    ],
    frames=[go.Frame(data=[
        go.Scatter(),
        go.Scatter(),
        go.Scatter(),
        go.Scatter(x=[samples_distances[i]], y=[sampled_energies[i]]),
        go.Scatter(x=[atom_a_x[i]]),
        go.Scatter(x=[atom_b_x[i]])
    ]) for i, _ in enumerate(samples_distances)]
)

fig.update_layout(
    title=dict(text=r"U(r) = 0.5 * k(r - r_eq)^2",
               x=0.5, xanchor="center", y=0.97, yanchor="top"),
    margin=dict(t=110),
    xaxis=dict(title="Bond Length (Angstroms)", range=[1.8, 2.2]),
    xaxis2=dict(title="Position: x", range=[0.9, 3.1], side='top', overlaying='x', showgrid=False),
    yaxis=dict(title="Energy (kcal/mol)", range=[0.0, 4.0]),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=25, redraw=False))])])]
)

fig.show()

In [89]:
bond_stretch_ene = compute_bond_stretch(distances, k=100)

fig = go.Figure(
    data=[
        go.Scatter(x=distances, y=v_curve, mode='lines', line=dict(color='gray', dash='dash'), name='Reference'),
        go.Scatter(x=distances, y=bond_stretch_ene, mode='lines', line=dict(color='red', dash='dash'), name='Harmonic Potential'),
        go.Scatter(x=samples_distances, y=sampled_energies, mode='markers', name='Energy State'),
        go.Scatter(x=atom_a_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 1'),
        go.Scatter(x=atom_b_x, y=[0.1], mode='markers', marker=dict(size=18, color='blue'), xaxis='x2', name='Atom 2')
    ],
    frames=[go.Frame(data=[
        go.Scatter(),
        go.Scatter(),
        go.Scatter(x=[samples_distances[i]], y=[sampled_energies[i]]),
        go.Scatter(x=[atom_a_x[i]]),
        go.Scatter(x=[atom_b_x[i]])
    ]) for i, _ in enumerate(samples_distances)]
)

fig.update_layout(
    xaxis=dict(title="Bond Length (Angstroms)", range=[1.6, 5.0]),
    xaxis2=dict(title="Position: x", range=[0, 17.0], side='top', overlaying='x', showgrid=False),
    yaxis=dict(title="Energy (kcal/mol)", range=[-0.8, 20.0]),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=25, redraw=False))])])]
)

fig.show()



<span style="color: red;">**Where does the harmonic approximation fail?**</span><br>



<span style="color: Green;">**Three Body Term:**</span><br>
How do we describe the potential w.r.t. a bond angle? 

In [91]:
import numpy as np
import nglview as nv
from ase import Atoms
from rdkit import Chem
from rdkit.Chem import AllChem 
from rdkit.Chem import rdMolTransforms


mol = Chem.MolFromSmiles('O')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol)
base_conf = mol.GetConformer(0)

# Linear scan angle (Forward scan)
forward_scan = np.linspace(80.0, 120.0, 15)  # bend from 80.0 deg - 120 deg
single_sweep = np.concatenate([forward_scan, forward_scan[::-1]]) 
full_scan = np.tile(single_sweep, 2)

ase_frames = []
for i, angle in enumerate(full_scan):
    new_conf = Chem.Conformer(base_conf)
    rdMolTransforms.SetAngleDeg(new_conf, 1, 0, 2, angle)
    
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    positions = new_conf.GetPositions()
    ase_frames.append(Atoms(symbols=symbols, positions=positions))

trajectory = nv.ASETrajectory(ase_frames)
view = nv.NGLWidget(trajectory)
view.layout.height = '200px'   

view.center()
view

NGLWidget(layout=Layout(height='200px'), max_frame=59)


Again a harmonic potential what is typically used<br>

$U_{bend} = \frac{1}{2}k(\theta - \theta_{eq})^2$ <br>

**Where does the harmonic approximation fail for bond bending?**

If the atoms positions are provided in cartesian coordinates how would compute $\theta$ <br>

<span style="color: Green;">**Four Body Term:**</span><br>
How do we describe the potential w.r.t. a dihedral? 

In [ ]:
import numpy as np
import nglview as nv
from ase import Atoms
from rdkit import Chem
from rdkit.Chem import AllChem 
from rdkit.Chem import rdMolTransforms

peptide_smiles = "C[C@@H](C(=O)O)N"
mol = Chem.MolFromSmiles(peptide_smiles)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol)
base_conf = mol.GetConformer(0)

# Linear scan torsion (Forward scan)
forward_scan = np.linspace(60.0, 160.0, 30) 
single_sweep = np.concatenate([forward_scan, forward_scan[::-1]]) 
full_scan = np.tile(single_sweep, 2)

ase_frames = []
for i, angle in enumerate(full_scan):
    new_conf = Chem.Conformer(base_conf)
    rdMolTransforms.SetDihedralDeg(new_conf, 2, 1, 5, 8, angle)
    
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    positions = new_conf.GetPositions()
    ase_frames.append(Atoms(symbols=symbols, positions=positions))

trajectory = nv.ASETrajectory(ase_frames)
view = nv.NGLWidget(trajectory)
view.layout.height = '200px'   

view.center()
view


## Dihedral 

Uses a Fourier Series 

$U_{dihedral} = \frac{V_1}{2}(1 + cos (\phi)) + \frac{V_2}{2}(1 - cos (2\phi)) + \frac{V_3}{2}(1 + cos (3\phi)) +\frac{V_4}{2}(1 - cos (4\phi))$ .... +


In [ ]:
import matplotlib.pyplot as plt

def compute_dihedral_energy(angles, v1=2.0, v2=3.0, v3=0.0, v4=0.0):
    v1_term = (v1/2.) * (1 + np.cos(angles))
    v2_term = (v2/2.) * (1 - np.cos(2.0*angles))
    v3_term = (v3/2.) * (1 + np.cos(3.0*angles))
    v4_term = (v4/2.) * (1 - np.cos(4.0*angles))
    return v1_term + v2_term + v3_term + v4_term

angles = np.linspace(0, 180, 360)
radians = np.deg2rad(angles)
pot1 = compute_dihedral_energy(radians, v1=1.0, v2=0.0, v3=0.0, v4=0.0)
v1 = pot1 - np.min(pot1)
pot2 = compute_dihedral_energy(radians, v1=0.0, v2=1.0, v3=0.0, v4=0.0)
v2 = pot2 - np.min(pot2)
pot3 = compute_dihedral_energy(radians, v1=0.0, v2=0.0, v3=1.0, v4=0.0)
v3 = pot3 - np.min(pot3)
pot4 = compute_dihedral_energy(radians, v1=0.0, v2=0.0, v3=0.0, v4=1.0)
v4 = pot4 - np.min(pot4)

total = (v1+v2+v3+v4)

fig, ax1 = plt.subplots()

ax1.set_xlabel('$\phi^{\circ}$')
ax1.set_ylabel('Term Energy  (kcal/mol)')
a1l1 = ax1.plot(angles, v1, linewidth=4.0, alpha=0.4, label='V1')
a1l2 = ax1.plot(angles, v2, linewidth=4.0, alpha=0.4, label='V2')
a1l3 = ax1.plot(angles, v3, linewidth=4.0, alpha=0.4, label='V3')
a1l4 = ax1.plot(angles, v4, linewidth=4.0, alpha=0.4, label='V4')

ax2 = ax1.twinx()
ax2.set_ylabel('Total Energy (kcal/mol)')
a2l = ax2.plot(angles, total, linewidth=2.0, alpha=0.99, color='black', label='Total')
lns = a1l1 + a1l2 + a1l3 + a1l4 + a2l
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs)


<span style="color: red;">**Why truncate at the $V_4$ term?**</span><br>

---


## Nonbonded Terms (Weak/ Long Range):
- Electrostatics
- Van der Waals interactions

---


#### **Electrostatics**


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.constants import physical_constants
from scipy.ndimage import map_coordinates
from skimage import measure

BOHR = physical_constants["Bohr radius"][0] * 1e10


def read_cube(path):
    """-> (data, origin, voxel_vectors, atom_xyz), atomic units."""
    with open(path) as f:
        head = [f.readline() for _ in range(6)]
        nat = abs(int(head[2].split()[0]))
        grid = np.array([ln.split() for ln in head[3:6]], float)
        atoms = np.array([f.readline().split() for _ in range(nat)], float)
        data = np.array(f.read().split(), float).reshape(grid[:, 0].astype(int))
    return data, np.array(head[2].split()[1:4], float), grid[:, 1:], atoms[:, 2:5]


def esp_surface(density_cube, esp_cube, isoval=0.002, opacity=1.0):
    dens, origin, vecs, atoms = read_cube(density_cube)
    verts, tris, *_ = measure.marching_cubes(dens, level=isoval)     # shape from density
    esp = map_coordinates(read_cube(esp_cube)[0], verts.T, order=1)  # colour from ESP
    xyz = (origin + verts @ vecs) * BOHR
    lim = np.percentile(np.abs(esp), 98)

    fig = go.Figure(go.Mesh3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        i=tris[:, 0], j=tris[:, 1], k=tris[:, 2],
        intensity=esp, colorscale='Turbo_r', cmin=-lim, cmax=lim, opacity=opacity,
        colorbar_title="ESP<br>(a.u.)"))
    fig.update_layout(width=700, height=600, scene=dict(
        aspectmode="data", xaxis_visible=False,
        yaxis_visible=False, zaxis_visible=False))
    return fig


esp_surface("benzene_density.cube", "benzene_esp.cube").show()

### Multipole expansion: Approximate Potential @ Long Distances
 
#### Coulomb's Law: 
- **Point Charges**<br>
$V(r) = \frac{1}{4\pi\varepsilon_{0}} \frac{q_j}{|r - r_{j}|}$ <br><br>
$U(r) = \frac{1}{4\pi\varepsilon_{0}} \frac{q_i q_j}{r_{ij}}$<br> <br>
$|{F}| \propto \frac{q_i q_j}{r_{ij}^2} $<br> <br>
$\bold{F}(r) = \frac{1}{4\pi\varepsilon_{0}} \frac{q_i q_j \hat{r}_{ij}}{r_{ij}^2}$ <br><br>


- **Continuous charge distribution $\rho(\bold r)$** <br>
$V(r) = \frac{1}{4\pi\varepsilon_{0}} \int_{V'} \frac{\rho(\bold r')}{|\bold{r} - \bold{r'}|} dV'$

**The multiple expansion is a series expansion that constructs a hierarchy of charge distributions whose fields fall off faster and faster.**

$V(r) = V_{monopole}(r) + V_{dipole}(r) + V_{quadrupole}(r) + ....$

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

K, D, QT = 332.0637, 1.0, -1.0        # kcal·Å/(mol·e²), charge spacing (Å), observer charge (e)

# Each multipole is built by cancelling the previous moment: first the net charge, then the dipole.
MULTIPOLES = [
    ("Monopole",   r"$1/r$",   np.array([1.0]),            np.array([[0.0, 0.0]])),
    ("Dipole",     r"$1/r^2$", np.array([1.0, -1.0]),      np.array([[0.0, D], [0.0, -D]])),
    ("Quadrupole", r"$1/r^3$", np.array([1.0, -2.0, 1.0]), np.array([[0.0, D], [0.0, 0.0], [0.0, -D]])),
]
CLR = ["tab:green", "tab:blue", "tab:red"]


def potential(q, pos, R):
    """Coulomb potential (kcal/mol per unit test charge) of charges q at pos, seen from points R.

    R holds the observation points with their (x, y) coordinates in the last axis.
    """
    r = np.linalg.norm(R[..., None, :] - pos, axis=-1)
    return K * (q / np.maximum(r, 0.3)).sum(-1)


L = 5.0                                                   
g = np.linspace(-L, L, 300)
GRID = np.stack(np.meshgrid(g, g), -1)                    
r_ax = np.linspace(3.0, 30.0, 300)                        
Y_AX = np.stack([np.zeros_like(r_ax), r_ax], -1)          

fig = plt.figure(figsize=(13, 8.5))
gs = fig.add_gridspec(2, 3, hspace=0.18, wspace=0.25)
maps = [fig.add_subplot(gs[0, i]) for i in range(3)]

for ax, (name, scaling, q, pos,) in zip(maps, MULTIPOLES):
    V = potential(q, pos, GRID)
    im = ax.imshow(V, extent=[-L, L, -L, L], origin="lower", cmap="RdBu",        
                   norm=SymLogNorm(linthresh=10, vmin=-500, vmax=500))
    ax.contour(g, g, V, levels=[-100, -40, -15, 15, 40, 100], colors="k", linewidths=0.5, alpha=0.4)
    ax.scatter(*pos.T, s=300, c=np.sign(q), cmap="coolwarm_r", vmin=-1, vmax=1, edgecolors="k", zorder=3)
    for qi, (xi, yi) in zip(q, pos):
        ax.text(xi, yi, f"{qi:+.0f}", ha="center", va="center", fontsize=9, zorder=4)
    ax.set_title(f"{name}    {scaling}", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])


# Walk a -1e observer out along the +y axis: U = q_observer * V, so every curve is attractive.
a = fig.add_subplot(gs[1, :])
for (name, scaling, q, pos), c in zip(MULTIPOLES, CLR):
    U = QT * potential(q, pos, Y_AX)
    a.plot(r_ax, U, c, lw=2.3, label=f"{name}   ({scaling})")
a.axhline(0, color="k", lw=0.8)
a.set_xlabel("Distance (Angstroms)")
a.set_ylabel("U(r) (kcal/mol)")
a.legend(fontsize=9, loc="lower right"); a.grid(alpha=0.3)

plt.show()


| Type of Interaction | Equation | Order of Magnitude (kcal/mol) |
| ------------------- | -------- | ------------------ |
| Ion-ion (charge-charge) | $\frac{1}{4\pi\varepsilon_{0}} \frac{q_i q_j}{r_{ij}}$ | +/- 10s to 100s |
| Ion-dipole | $\frac{1}{4\pi\varepsilon_{0}} \frac{\mu q \theta}{r^2}$ | +/- 1 to 10 |
| Dipole-Dipole | $\frac{1}{4\pi\varepsilon_{0}} \frac{\mu_i \mu_j}{r_{ij}^3} (\cos\theta_{ij} - \cos\theta_{i}\cos\theta_{j})$ | +/- 1 to 5 |
| Ion-induce dipole | $\frac{1}{2(4\pi\varepsilon_{0})^2} \frac{\alpha q^2}{r^4}$  | +/- 1 to 5 |

#### How do we simplify a complicated charge distribution ?

![esp2q](../assets/images/ESP_2_point_charges.png)

- **Construct as set of atomic partial charges that approximates the QM ESP**
  - **The sum of atomic charges equals the total formal charge of the molecule**
  - **Mean value approximation**
    - **Electron density will change wrt geometry** 
    - **Electron density will change wrt electric field (polarization)**

---

#### **Van der Walls Interactions**

**Lennard-Jones or 12-6 potential**

$U_{LJ}(r) = 4\epsilon[(\frac{\sigma}{r})^{12} - (\frac{\sigma}{r})^6]$ <br><br>

$\epsilon =$ the depth of the potential well <br>
$\sigma =$ distance at which the atom-atom potential energy is zero


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

def compute_r12_term(r):
    return 12.0*(2.0/r)**12

def compute_r6_term(r):
    return -12.0*(2.0/r)**6

def compute_lj_potential(r):
    v = 4.0*3.0*((2.0/r)**12 - (2.0/r)**6)
    return v

distances = np.linspace(1.8, 8.0, 100)
lj = compute_lj_potential(distances)
r6 = compute_r6_term(distances)
r12 = compute_r12_term(distances)

plt.plot(distances, lj, '-', color='black', linewidth=4.0, label='LJ')
plt.plot(distances, r6, '--', color='red', label=r'$r^6$ Term')
plt.plot(distances, r12, '--', color='blue', label=r'$r^{12}$ Term')
plt.ylim(-21.0, 21.0)
plt.xlabel('Distance (Angstroms)')
plt.ylabel('Energy (kcal/mol)')
plt.legend()


### Classical Force Field Potential

$U_{Total} = \displaystyle \sum_{i<j}4\epsilon[(\frac{\sigma}{r_{ij}})^{12} - (\frac{\sigma}{r_ij})^6] + \sum_{i<j} \frac{1}{4\pi\varepsilon_{0}} \frac{q_i q_j}{r_{ij}} + \sum_{bonds} k_r(r - r_{eq})^2 + \sum_{angles} k_{\theta}(\theta - \theta_{eq})^2 + \sum_{dihedrals} + \frac{V_1}{2}(1 + cos (\phi)) + \frac{V_2}{2}(1 - cos (2\phi)) + \frac{V_3}{2}(1 + cos (3\phi)) +\frac{V_4}{2}(1 - cos (4\phi))$

---

# Introduction to Molecular Dynamics and Statistical Mechanics 

---

### With the potential energy function we can simulate dynamic and thermodynamic properties of proteins, nucleic acids, other biologically relevant molecules.

### **Netwon's Second Law: Force**
$\bold F = m\mathbf{a} = \frac{\partial \mathbf{p}}{\partial t} = -\nabla U(r)$

$\nabla f(\mathbf{x_{n}}) = \frac{\partial f}{\partial x}\mathbf{i} + \frac{\partial f}{\partial y}\mathbf{j} + \frac{\partial f}{\partial z}\mathbf{k}$

where **i**,**j**,**k** are the standard unit vectors in the direction of x,y,z 

#### Numerical method for integration Newton's equation of motion:<br>
<u>Verlet integration</u><br>
$\mathbf{x_{n+1}} = 2\mathbf{x_{n}} - \mathbf{x_{n-1}} + \mathbf{a_n}\Delta t^2$

### Common Statistical Ensembles:
- **NVE**: constant-energy, constant-volume, also known as the microcanonical ensemble. 
  - Solve Newton's equation of motion without temperature and pressure 
- **NVT**: constant-temperature, constant-volume, also known as the canonical ensemble
  - The temperature is controlled by a thermostat (regulates the velocities)
- **NPT**: constant-temperature and constant-pressure
  - The pressure is controlled by a barostat (regulates the volume)  
  
---

### Putting it Together: an NVE Simulation of CO on a Cu(111) Surface

Everything needed to run a simulation is now on the table:
1. **A potential energy function** $U(\mathbf{r})$ &mdash; here the UMA potential is used
2. **Forces** from $-\nabla U(\mathbf{r})$
3. **Velocity Verlet Algorithm**, so we can integrator Newton's equation of motion


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from ase import units
from ase.optimize import BFGS
from ase.calculators.emt import EMT
from ase.io import Trajectory, read
from ase.md.verlet import VelocityVerlet
from ase.constraints import FixAtoms, FixBondLength
from ase.build import fcc111, add_adsorbate, molecule
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

# 4-layer Cu(111) slab, periodic in x and y, with 10 Angstroms of vacuum above
slab = fcc111('Cu', size=(3, 3, 4), vacuum=10.0)

co = molecule('CO')
co.rotate(180, 'x')                                    # point the carbon down at the surface
add_adsorbate(slab, co, height=2.0, position='ontop')

# Freeze the bottom two layers so they behave like bulk metal, and hold the C-O bond
# rigid because EMT has no usable intramolecular potential for the molecule
slab.set_constraint([FixAtoms(mask=[atom.tag >= 3 for atom in slab]),
                     FixBondLength(len(slab) - 2, len(slab) - 1)])
slab.calc = EMT()

BFGS(slab, logfile=None).run(fmax=0.05)                # relax into the nearest minimum before any dynamics

print(f'{len(slab)} atoms, {slab.get_number_of_degrees_of_freedom()} moving degrees of freedom')
print(f'U = {slab.get_potential_energy():.3f} eV')

In [ ]:
def wrap_adsorbate(atoms):
    """Slide CO back into the periodic cell as a unit, so it stays in view."""
    atoms = atoms.copy()
    cell = atoms.cell[:2]                                            # in-plane lattice vectors
    n = np.floor(np.linalg.solve(cell[:, :2].T, atoms.positions[-2, :2]))
    atoms.positions[-2:] -= n @ cell
    return atoms


def show(images):
    """Render the slab space-filling with the adsorbate as ball-and-stick."""
    if not isinstance(images, list):
        images = [images]
    view = nv.show_asetraj([wrap_adsorbate(atoms) for atoms in images])
    view.clear_representations()
    view.add_spacefill(selection='_Cu', radius=1.3)
    view.add_ball_and_stick(selection='not _Cu')
    view.add_unitcell()
    view.center()
    return view


show(slab)

In [ ]:
T_target = 300        # K
timestep = 1.0        # fs
n_steps = 40_000     # 100 ps of dynamics -- this takes a couple of minutes to run

MaxwellBoltzmannDistribution(slab, temperature_K=T_target)

# friction = 0.01/fs, i.e. the bath exchanges energy with the atoms on a ~100 fs timescale
dyn = VelocityVerlet(slab, timestep=timestep * units.fs)

traj = Trajectory('co_cu111_nvt.traj', 'w', slab)
dyn.attach(traj.write, interval=200)                        # a frame every 200 fs -> 500 frames

log = []
dyn.attach(lambda: log.append((dyn.get_time() / units.fs, slab.get_temperature(),
                               slab.get_potential_energy(), slab.get_kinetic_energy())),
           interval=50)
dyn.attach(lambda: print(f'{dyn.get_time() / (1000 * units.fs):4.0f} ps'
                         f'   T = {slab.get_temperature():6.1f} K'), interval=10000)

dyn.run(n_steps)
traj.close()

time_ps, temperature, e_pot, e_kin = np.array(log).T
time_ps = time_ps / 1000
frames = read('co_cu111_nvt.traj', ':')
print(f'{len(frames)} frames saved over {time_ps[-1]:.0f} ps')

In [ ]:
show(frames)

In [ ]:
frame_ps = np.linspace(0, time_ps[-1], len(frames))    # frames are saved less often than the log
equil = time_ps > 1            # throw away the first 1 ps while the system settles
T_mean, T_std = temperature[equil].mean(), temperature[equil].std()

fig, ax = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

ax[0].plot(time_ps, temperature, color='gray', linewidth=0.5)
ax[0].axhline(T_target, color='red', linestyle='--', label=f'Target = {T_target} K')
ax[0].axhline(T_mean, color='blue', label=f'Mean = {T_mean:.0f} K')
ax[0].set_ylabel('T (K)')
ax[0].legend()

# distance from the oxygen to whichever Cu atom is nearest, measured across the periodic boundary
oxygen = [atom.index for atom in slab if atom.symbol == 'O'][0]
copper = [atom.index for atom in slab if atom.symbol == 'Cu']
cu_o = np.array([atoms.get_distances(oxygen, copper, mic=True).min() for atoms in frames])

ax[1].plot(frame_ps, cu_o, color='purple', linewidth=0.8)
ax[1].axhline(cu_o.mean(), color='red', linestyle='--', label=f'Mean = {cu_o.mean():.2f} A')
ax[1].set_ylabel('Nearest Cu-O distance (A)')
ax[1].legend()

ax[2].plot(time_ps, e_pot - e_pot[0], color='black', label='Potential')
ax[2].plot(time_ps, e_kin, color='orange', label='Kinetic')
ax[2].set_xlabel('Time (ps)')
ax[2].set_ylabel('Energy (eV)')
ax[2].legend()
plt.tight_layout()

<span style="color: red;">**Does the force field method accurately model structural, dynamic, and thermodynamic properties (e.g. CO binding to copper)?**.</span><br>

---

## Statistical Mechanics 

Now that we defined the energy potential and how to simulate the dynamics, what are the energy states that are actually sampled? How does this affect thermodynamic properties.<br><br>

For thermodynamic properties, we are talking about a very large number of molecules in or near states of equilibrium. That means we don't need to "follow" one molecule but calculate the ensemble average from a statistical point a view. <br> <br>

This mean we should look for the most probable distribution of energy, as for we do not expect an equilibrium state to be improbable one.

---
### Counting Microstates: Where the Boltzmann Distribution Comes From

The simulation samples configurations, but why does it visit some far more often than others? Fix the number of particles and the total energy, and count.

For $N$ particles spread over energy states with $n_1$ in the first, $n_2$ in the second, and so on, the number of distinct arrangements is

$W = \frac{N!}{n_1!\, n_2!\, n_3! \cdots}$

Below are three ways that six particles can share a total energy of $10\epsilon$. All three satisfy the same two constraints &mdash; same $N$, same $E$ &mdash; but they are not equally likely.

<span style="color: red;">**Which distribution would you expect to find the system in, and why?**</span>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial

# each entry maps an energy level (in units of epsilon) to the number of particles sitting in it
distributions = [{1: 5, 5: 1}, {1: 3, 2: 2, 3: 1}, {1: 4, 3: 2}]
levels = range(1, 7)


def multiplicity(occupancy):
    """W = N! / (n_1! n_2! ...): the number of ways to arrange labelled particles into these levels."""
    w = factorial(sum(occupancy.values()))
    for n in occupancy.values():
        w //= factorial(n)
    return w


fig, axes = plt.subplots(1, 3, figsize=(11, 4.5), sharey=True)

for ax, occupancy, panel in zip(axes, distributions, 'abc'):
    for level in levels:
        ax.hlines(level, 0.0, 1.0, color='black', linewidth=1.2)
    for level, n in occupancy.items():
        x = 0.5 + 0.1 * (np.arange(n) - (n - 1) / 2)       # centre the particles on the line
        ax.plot(x, [level] * n, 'o', color='black', markersize=9)
    ax.set_title(f'{multiplicity(occupancy)} permutations')
    ax.set_xlabel(f'({panel})')
    ax.set_xticks([])
    ax.set_xlim(-0.1, 1.1)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)

axes[0].set_yticks(list(levels), [f'{i}$\\epsilon$' for i in levels])
fig.suptitle('$N = 6$ particles sharing a fixed total energy $E = 10\\epsilon$')
plt.tight_layout()

Distribution **(b)** wins by a factor of 4 over (c) and 10 over (a). Maximizing $W$ subject to fixed $N$ and fixed $E$ (two constraints, two Lagrange multipliers) gives the Boltzmann distribution. <br>

$n_i = n_1 e^{-(\epsilon_i - \epsilon_1) / k_{B}T}$<br>

$k_B = \frac{R}{N_A}$ 

Account for energy levels and degeneracy:<br><br>
$n_i = \frac{g_i}{g_1}n_1 e^{-(\epsilon_i - \epsilon_1) / k_{B}T}$ <br><br><br>


Free energy and entropy can be defined as<br><br> 

$\frac{n_i}{n_j} = \frac{g_i}{g_1}n_1 e^{-(\epsilon_i - \epsilon_j) / k_{B}T} = e^{-\Delta G/RT}$<br><br>
$\ln(\frac{n_i}{n_j}) = -\frac{\Delta G}{RT}$<br>

$S=k_B lnW$



---


### Statistical Weights and the Partition Function.

The simplest multistate is one which a molecule can exist in one of two states (A or B).<br><br>
**A <--> B** <br>
$K_{eq} = [B]/[A]$ <br><br>
What is the probability of observing molecule B relative to the probability of observing A. This is by definition $K_{eq}$. More generally, the probability of finding B versus all possible states $P_B = [B]/\sum[state_i]$<br><br> 
$P_B = \frac{[B]}{[A] + [B]} = \frac{\frac{[B]}{[A]}}{\frac{[A]}{[A]} + \frac{[B]}{[A]}} = \frac{K_{eq}}{1 + K_{eq}}$

$P_B = \frac{\omega_B}{\omega_B + \omega_A} = \frac{\omega_B}{1 + \omega_B}$<br><br>
where<br>
$\omega_A = [A]/[A]$ and $\omega_B = [B]/[A]$<br><br>

when treated as an equilibrium constant, $\omega$ is related to the difference in free energy between states.<br><br>
$\Delta G_j = -RT\ln\omega_j$ <br>
or<br>
$\omega_j = e^{-\Delta G_j/RT}$<br><br><br>

For **A <--> B**:<br>
$P_B = \frac{e^{-\Delta G_B/RT}}{1 + e^{-\Delta G_B/RT}}$<br><br><br>



**partition function**:<br> 
$Q = \sum_j e^{-\epsilon_j / k_BT}$<br>
or<br>
$Q = \sum_j g_j\omega_j$

---

### Double-Well System:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

# 1. Convert Boltzmann constant (J/K) to molar gas constant R (J/mol·K)
R_joules =  constants.k * constants.N_A

# 2. Convert Joules to kilocalories (using thermochemical calorie 'cal')
# const.calorie is equal to 4.184 Joules
Kb = R_joules / (constants.calorie * 1000)

print(f"Boltzmann constant in kcal/(mol·K): {Kb:.9f}")
TEMP = 300.0 # Temperature in Kelvin
BETA = 1.0 / (Kb * TEMP)


def compute_double_well_potential(x, v_params):
    """
    Computes the double well potential V(x) = a * x^4 - b * x^2 + c * x^3.
    
    Parameters:
    x : array-like
        The input values for which to compute the potential.
    a : float
        Coefficient for the x^4 term.
    b : float
        Coefficient for the x^2 term.
    c : float
        Coefficient for the x^3 term.
    
    Returns:
    array-like
        The computed potential values.
    """
    a, b, c = v_params
    return a * x**4 - b * x**2 + c * x**3

def run_mcmc_sampling(x_start, beta, num_samples=10000, proposal_std=0.5, v_params=(1.0, 2.0, 0.7)):
    """
    Runs MCMC sampling using the Metropolis-Hastings algorithm.
    
    Parameters:
    v : array-like
        The potential energy values corresponding to the input x values.
    x_start : float
        The starting point for the MCMC sampling.
    beta : float
        The inverse temperature (1/(k_B * T)).
    num_samples : int
        The number of samples to generate.
    proposal_std : float
        The standard deviation of the proposal distribution.
    
    Returns:
    list
        A list of sampled x values.
    """
    samples = []
    current_x = x_start
    current_v = compute_double_well_potential(current_x, v_params)
    
    for _ in range(num_samples):
        # Propose a new sample from a normal distribution centered at current_x
        proposed_x = np.random.normal(current_x, proposal_std)
        
        # Compute the potential at the proposed point
        proposed_v = compute_double_well_potential(proposed_x, v_params)
        
        # Compute acceptance probability
        acceptance_prob = min(1.0, np.exp(-beta * (proposed_v - current_v)))
        
        # Accept or reject the proposed sample
        if np.random.rand() < acceptance_prob:
            current_x = proposed_x
            current_v = proposed_v
        
        samples.append(current_x)
    
    return samples

x = np.linspace(-2.0, 2.0, 150)
params = (1.7, 5.0, 0.0)  # a, b, c parameters for the double well potential
v = compute_double_well_potential(x, params)

x_start = x[np.argmin(v)]
samples = run_mcmc_sampling(x_start, BETA, num_samples=10000, proposal_std=0.5, v_params=params)
rel_v = v - np.min(v)

bin_width = 0.025
bin_edges = np.arange(min(samples), max(samples) + bin_width, bin_width)

plt.clf()
fig, ax1 = plt.subplots(layout='constrained')
ax2 = ax1.twinx()
ax1.plot(x, rel_v, label='Double Well Potential')
ax2.hist(samples, bins=bin_edges, alpha=0.5, label='MCMC Samples')
ax1.set_xlabel('X')
ax1.set_ylabel('Energy (kcal/mol)')
ax2.set_ylabel('Counts')
ax1.legend()
plt.show()

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

plt.clf()
rel_v = v - np.min(v)
frames = 500 # Match frames to FuncAnimation frames
data = np.array_split(samples, frames)

bin_width = 0.025
bin_edges = np.arange(min(samples), max(samples) + bin_width, bin_width)

fig, ax1 = plt.subplots(layout='constrained')
ax2 = ax1.twinx()

ax1.set_xlabel('X')
ax1.set_ylabel('Energy (kcal/mol)', color='tab:red')
ax2.set_ylabel('Counts', color='tab:blue') # Added for clarity

line1, = ax1.plot([], [], color='tab:red', lw=2)
line2, = ax2.plot([], [], color='tab:blue', lw=2)

# --- CRITICAL FIX: Set limits manually for blitting ---
# Set X limits based on your x array
ax1.set_xlim(np.min(x), np.max(x))
ax2.set_xlim(np.min(x), np.max(x))

# Set Y limits based on the maximum expected values of your data
ax1.set_ylim(np.min(rel_v) - 0.1, np.max(rel_v) + 1)
# Estimate a safe upper bound for your histogram density
ax2.set_ylim(0, 2.0) 
ax2.set_ylim(0, np.max(np.histogram(samples, bins=bin_edges)[0]) + 1)

# Create fixed bins so the histogram shape maps directly to your X coordinates
line1, = ax1.plot([], [], color='tab:red', lw=2)

# Create an empty histogram on ax2. This generates the actual bar patches.
# Matplotlib will handle all the spacing; you do not need 'bin_centers' anymore.
_, _, bar_container = ax2.hist([], bins=bin_edges, alpha=0.5, edgecolor='black', color='skyblue')

accumulated_samples = []

# --- 3. Animation Functions ---
def init():
    """Initialize the animation background."""
    line1.set_data([], [])
    
    # Set all histogram bar heights to 0 to clear the frame
    for patch in bar_container:
        patch.set_height(0)
        
    # When using blit=True, you must return a flat list of all visual elements
    return [line1] + list(bar_container)


def animate(i):
    # This remains untouched
    line1.set_data(x, rel_v)
    
    # Add new samples to your running total list
    accumulated_samples.extend(data[i])
    
    # Calculate the frequency counts for the histogram
    counts, _ = np.histogram(accumulated_samples, bins=bin_edges)
    
    # FIX: Update the actual height of each bar patch individually
    for patch, count in zip(bar_container, counts):
        patch.set_height(count)
        
    return [line1] + list(bar_container)

# --- 4. Create and Show the Animation ---
ani = animation.FuncAnimation(
    fig, 
    animate, 
    init_func=init, 
    frames=frames, 
    interval=50, # 200ms
    blit=True, 
    repeat=False
)

# ani.save("double_well_mcmc.gif", writer=animation.PillowWriter(fps=4)) 
plt.show()

### Ergodic hypothesis

A system evolving according to the hamilton's equations motion (reformulation of Newton's), given an infinite amount of time, the time average of a physical observable will equal the ensemble average.

$\beta = (k_BT)^{-1}$ <br><br>
$\displaystyle <A>_\beta = \frac{\int d\mathbf{q} e^{-\beta U(\mathbf{q})} A(\mathbf{q})}{\int d\mathbf{q} e^{\beta U(\mathbf{q})}}$ <br> <br>

$\displaystyle \bar{A} = \lim_{T \to \infty} \frac{1}{T} \int_{0}^{t} A(\mathbf{q})dt $


$\bar{A} = <A>$

